# 04 — Évaluation finale sur le jeu de test

Test = steps 632 à 743, la période la plus récente. Il n'a servi à aucun choix :
modèle, variables, poids, hyperparamètres et seuil ont tous été fixés sur la validation.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("fraude-04-test")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "spark-master")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .config("spark.sql.shuffle.partitions", "18")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", "file:///tmp/spark-events")
         .getOrCreate())

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 16:57:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

test = spark.read.parquet("/opt/spark-data/ml/test.parquet")
modele = PipelineModel.load("/opt/spark-data/modeles/modele_fraude")

with open("/opt/spark-data/modeles/seuil.txt") as f:
    seuil = float(f.read())

nb = test.count()
nb_fraudes = test.filter("label = 1").count()
print("Test :", nb, "lignes,", nb_fraudes, "fraudes")
print("PR-AUC du hasard (taux de fraude) :", round(nb_fraudes / nb, 4))
print("Seuil utilisé :", seuil)

Test : 88440 lignes, 1235 fraudes
PR-AUC du hasard (taux de fraude) : 0.014
Seuil utilisé : 0.7


88 440 transactions dont 1 235 fraudes (1,4 %). Un modèle au hasard obtiendrait une PR-AUC de 0,014.

In [3]:
# probabilité de fraude, puis prédiction avec le seuil choisi en validation
predictions = modele.transform(test)
predictions = predictions.withColumn("p_fraude", vector_to_array("probability")[1])
predictions = predictions.withColumn("prediction_seuil", (F.col("p_fraude") >= seuil).cast("double"))
predictions.cache()

pr_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(predictions)
print("PR-AUC :", round(pr_auc, 4))

for colonne in ["prediction", "prediction_seuil"]:
    print("\nPrédiction :", colonne)
    for metrique in ["precisionByLabel", "recallByLabel", "fMeasureByLabel"]:
        evaluateur = MulticlassClassificationEvaluator(labelCol="label", predictionCol=colonne,
                                                       metricName=metrique, metricLabel=1.0)
        print("  ", metrique, ":", round(evaluateur.evaluate(predictions), 4))

PR-AUC : 0.6712

Prédiction : prediction
   precisionByLabel : 0.8599
   recallByLabel : 0.4275
   fMeasureByLabel : 0.5711

Prédiction : prediction_seuil
   precisionByLabel : 0.9108
   recallByLabel : 0.3887
   fMeasureByLabel : 0.5448


- PR-AUC de 0,6712, soit environ 48 fois le hasard. En validation, c'était 0,6028 pour un hasard à 0,0062, soit environ 97 fois. La PR-AUC du test est plus haute seulement parce qu'il y a plus de fraudes dans cette période : par rapport au hasard, le modèle perd un peu en avançant dans le temps.
- Au seuil 0,7, la précision est de 0,91 et le rappel de 0,39.
- Ici, le seuil 0,5 aurait donné un F1 un peu meilleur (0,5711 contre 0,5448). Le seuil a été choisi sur la validation, où les fraudes étaient plus rares : quand la proportion de fraudes change, le meilleur seuil change aussi. On garde quand même 0,7, car le test ne doit servir à aucun choix.

## Matrice de confusion (lignes = vraie classe, colonnes = classe prédite)

In [4]:
predictions.groupBy("label").pivot("prediction_seuil", [0.0, 1.0]).count().na.fill(0).orderBy("label").show()

+-----+-----+---+
|label|  0.0|1.0|
+-----+-----+---+
|  0.0|87158| 47|
|  1.0|  755|480|
+-----+-----+---+



480 fraudes détectées sur 1 235 (755 manquées), et seulement 47 fausses alertes sur 87 205 transactions normales. Le modèle signale peu de transactions, mais ses alertes sont presque toujours justes.

## Résultats par type de transaction

In [5]:
(predictions.groupBy("type")
            .agg(F.count("*").alias("nb"),
                 F.sum("label").alias("fraudes"),
                 F.sum(F.col("prediction_seuil") * F.col("label")).alias("fraudes_detectees"),
                 F.sum(F.col("prediction_seuil") * (1 - F.col("label"))).alias("fausses_alertes"))
            .orderBy(F.desc("fraudes"))
            .show())

+--------+-----+-------+-----------------+---------------+
|    type|   nb|fraudes|fraudes_detectees|fausses_alertes|
+--------+-----+-------+-----------------+---------------+
|CASH_OUT|28824|  620.0|            252.0|           10.0|
|TRANSFER| 8711|  615.0|            228.0|           37.0|
|   DEBIT|  707|    0.0|              0.0|            0.0|
| CASH_IN|20484|    0.0|              0.0|            0.0|
| PAYMENT|29714|    0.0|              0.0|            0.0|
+--------+-----+-------+-----------------+---------------+



Toutes les fraudes sont dans `CASH_OUT` et `TRANSFER`, avec à peu près la même détection (252 sur 620 et 228 sur 615). Aucune fausse alerte sur les autres types : le modèle a bien appris le rôle de `type`.

## Importance des features

In [6]:
dernier_etage = modele.stages[-1]

# noms des features, lus dans les métadonnées du VectorAssembler
noms = {}
for groupe in predictions.schema["features_raw"].metadata["ml_attr"]["attrs"].values():
    for attribut in groupe:
        noms[attribut["idx"]] = attribut["name"]

if hasattr(dernier_etage, "featureImportances"):        # Random Forest / GBT
    valeurs = dernier_etage.featureImportances.toArray()
else:                                                   # régression logistique
    valeurs = dernier_etage.coefficients.toArray()

for i in sorted(noms, key=lambda i: -abs(valeurs[i])):
    print(f"{noms[i]:<20} {valeurs[i]:.4f}")

oldbalanceDest       0.3341
heure                0.2704
logAmount            0.1882
typeVec_CASH_OUT     0.1292
typeVec_TRANSFER     0.0589
typeVec_CASH_IN      0.0116
typeVec_PAYMENT      0.0076


`oldbalanceDest` (0,33), l'heure (0,27) et le montant (0,19) portent l'essentiel. Aucune variable n'écrase les autres, contrairement à l'ancienne version où l'écart de solde de l'émetteur pesait 50 %.

## Conclusion

- Les 11 problèmes des données brutes ont été corrigés avec Spark (SQL, DataFrame, `Imputer`).
- Pipeline final : `Imputer` → `SQLTransformer` → `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler` → GBT (60 itérations, profondeur 5), poids 10 pour une fraude, seuil 0,7.
- Variables : montant, type, heure, solde du destinataire avant la transaction. Aucune information d'après transaction, pas de raccourci du simulateur.
- Test sur la période la plus récente : PR-AUC 0,6712 (hasard 0,014), précision 0,91, rappel 0,39.
- Avec le raccourci (`oldbalanceOrg`), la PR-AUC montait à 0,83 en validation, et à 0,98 dans l'ancienne version : ces scores venaient de la règle du simulateur, pas d'une vraie détection.
- Limite : la proportion de fraudes change beaucoup entre les périodes de PaySim, ce qui rend le choix du seuil moins stable.

In [7]:
spark.stop()